# HRNet-W18 Jacquard V2 — Kaggle training

Runs on a Kaggle notebook with a **T4** or **P100** accelerator (Settings → Accelerator → GPU T4 x1 recommended).

## Dataset

Uses the public Kaggle dataset **[`vdsdggsgsd/jacquard`](https://www.kaggle.com/datasets/vdsdggsgsd/jacquard)** (~77 GB, Apache-2.0). It is **already unpacked** into ~11 600 object folders under `/kaggle/input/datasets/vdsdggsgsd/jacquard/Jacquard/`, so no download or unzip is needed in the notebook — Kaggle mounts the dataset straight from its CDN and DataLoader reads directly from there.

## One-time setup before opening this notebook on Kaggle

1. Confirm your account has GPU access (Settings → Account → Phone Verification).
2. https://www.kaggle.com/code → **+ New Notebook**.
3. File → **Import Notebook** → URL → paste `https://github.com/Hesgoryr/HRNet_Grasp_Semantic_Segmentation/blob/main/notebooks/kaggle_train.ipynb` (or File → Upload to upload a local copy).
4. Right panel → **+ Add Input** → search `vdsdggsgsd/jacquard` → **+ Add**.
5. Right panel → Settings → **Accelerator** = `GPU T4 x1`, **Internet** = On, **Persistence** = Files only.
6. Run All.

## What the notebook does
1. Clones the training code from GitHub into `/kaggle/working/`.
2. Installs `imagecodecs` (Kaggle preinstalls torch/timm/cv2/tifffile/numpy).
3. Builds an object-wise train/val/test split over `/kaggle/input/datasets/vdsdggsgsd/jacquard/Jacquard/`.
4. Trains HRNet-W18 angle-segmentation (RGB-only by default; switch `CONFIG_PATH` to `configs/default.yaml` for RGB-D) with AMP on T4; checkpoints land in `/kaggle/working/runs/<run_name>/` and persist as the notebook output when you Save Version.
5. Auto-resumes if a previous run output is attached as another input (see step 5).
6. Plots training curves from `metrics.csv`.

## Session limits
Kaggle caps GPU usage at **30 h/week** and **12 h/session** (with auto-shutdown after 40 min of inactivity). A full 80-epoch run takes ~12–15 h on T4 — i.e. one or two sessions. Use the resume flow in step 5 to continue across sessions.

## 0. User-supplied configuration

In [ ]:
# ====== EDIT THESE ======
# Path to the object folders (pre-unpacked by the dataset author).
DATA_DIR = "/kaggle/input/datasets/vdsdggsgsd/jacquard/Jacquard"
# Training config — pick one:
#   configs/rgb.yaml      → 3 input channels (RGB only), no depth
#   configs/default.yaml  → 4 input channels (RGB + depth)
CONFIG_PATH = "configs/rgb.yaml"
RUN_NAME = "hrnet_w18_rgb_angle"  # rename if you switch CONFIG_PATH
GITHUB_REPO = "https://github.com/Hesgoryr/HRNet_Grasp_Semantic_Segmentation.git"
GITHUB_BRANCH = "main"
# Optional: attach a previous version of this notebook's output as an input dataset to auto-resume.
# Set RESUME_INPUT_DIR to the path where it was mounted (e.g. "/kaggle/input/hrnet-resume/runs/<RUN_NAME>").
RESUME_INPUT_DIR = None
# Training overrides (T4 has 16 GB VRAM, P100 has 16 GB; bs=16 fits on both with AMP).
IMAGE_SIZE = 384
BATCH_SIZE = 16
ACCUM_STEPS = 1
NUM_WORKERS = 4
EPOCHS = 80
# ========================

## 1. Verify GPU and dataset

In [ ]:
!nvidia-smi

In [ ]:
import os
assert os.path.isdir(DATA_DIR), f'dataset not found at {DATA_DIR}. Did you attach vdsdggsgsd/jacquard in Settings → + Add Input?'
objs = [f for f in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, f))]
print(f'found {len(objs)} object folders')
# quick sanity: confirm file names look like V2 layout
sample_obj = os.path.join(DATA_DIR, objs[0])
print('sample files in', objs[0], ':', sorted(os.listdir(sample_obj))[:6])

## 2. Clone repo + install missing deps

Kaggle preinstalls torch/timm/cv2/tifffile. We only need `imagecodecs` (LZW depth TIFFs).

In [ ]:
REPO_DIR = '/kaggle/working/HRNet_Grasp_Semantic_Segmentation'
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 -b {GITHUB_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch --depth 1 origin {GITHUB_BRANCH} && git reset --hard origin/{GITHUB_BRANCH}
%cd {REPO_DIR}
!pip install -q imagecodecs

In [ ]:
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 3. Build the object-wise split

`/kaggle/input/` is read-only, so we write the split into the repo checkout under `/kaggle/working/` (both `--resume` and the training step read from there). Re-runs within the same session reuse the existing split file.

In [ ]:
SPLITS_PATH = os.path.join(REPO_DIR, 'splits/jacquard_v2.json')
if not os.path.exists(SPLITS_PATH):
    !python tools/prepare_split.py --root {DATA_DIR} --out {SPLITS_PATH} --val-frac 0.1 --test-frac 0.1 --seed 0
else:
    print('split already exists, reusing:', SPLITS_PATH)

## 4. Optional — copy a previous run's checkpoint to resume

Kaggle notebook sessions cap at **12 h** and **wipe `/kaggle/working/`** between sessions. To continue past one session:
1. Run the notebook until `last.pth` exists in `/kaggle/working/runs/<RUN_NAME>/`.
2. Click **Save Version** (top-right). After the version finishes, the working folder becomes the notebook's downloadable output.
3. Either use the notebook output directly as input next time, or create a new private Kaggle Dataset from it.
4. In the next session, attach that output/dataset in Settings → + Add Input, then set `RESUME_INPUT_DIR` above to the mounted path that contains `last.pth`.

In [ ]:
import shutil
SAVE_DIR = f'/kaggle/working/runs/{RUN_NAME}'
os.makedirs(SAVE_DIR, exist_ok=True)

if RESUME_INPUT_DIR and os.path.isdir(RESUME_INPUT_DIR):
    print(f'copying previous run from {RESUME_INPUT_DIR} → {SAVE_DIR}')
    for fname in os.listdir(RESUME_INPUT_DIR):
        src = os.path.join(RESUME_INPUT_DIR, fname)
        dst = os.path.join(SAVE_DIR, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    print('done')
else:
    print('no previous run attached; starting fresh')

## 5. Train

In [ ]:
RESUME_PATH = os.path.join(SAVE_DIR, 'last.pth')
resume_arg = f'--resume {RESUME_PATH}' if os.path.exists(RESUME_PATH) else ''
cmd = (f'python tools/train.py --config {CONFIG_PATH} '
       f'dataset.root={DATA_DIR} '
       f'dataset.splits_path={SPLITS_PATH} '
       f'dataset.image_size={IMAGE_SIZE} '
       f'dataset.num_workers={NUM_WORKERS} '
       f'trainer.epochs={EPOCHS} '
       f'trainer.batch_size={BATCH_SIZE} '
       f'trainer.accum_steps={ACCUM_STEPS} '
       f'trainer.save_dir={SAVE_DIR} '
       f'{resume_arg}')
print(cmd)
!{cmd}

## 6. Plot training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(os.path.join(SAVE_DIR, 'metrics.csv'))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if 'train_loss' in df.columns:
    axes[0].plot(df['epoch'], df['train_loss'], label='train_loss')
axes[0].set_title('train loss'); axes[0].legend(); axes[0].grid()
for col in ['val_miou_fg', 'val_dice_fg']:
    if col in df.columns:
        axes[1].plot(df['epoch'], df[col], label=col)
axes[1].set_title('val metrics'); axes[1].legend(); axes[1].grid()
plt.tight_layout(); plt.show()